# 👑 Diya Rai — 1-Click SDXL Face LoRA Training
**Google Colab Free T4 GPU (~20 mins)**
Train your custom `diyarai_sdxl_lora.safetensors` model to generate Diya Rai in Fooocus, ComfyUI, or Automatic1111 with 100% facial consistency!

In [ ]:
# Step 1: Install Kohya SD-Scripts & PyTorch dependencies
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q --upgrade -r requirements.txt
!pip install -q xformers bitsandbytes accelerate diffusers
print('✅ Dependencies Installed Successfully!')

In [ ]:
# Step 2: Upload and Extract Diya Rai Dataset
import os, zipfile
%cd /content
os.makedirs('/content/dataset', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

if not os.path.exists('/content/diyarai_lora_dataset.zip'):
    from google.colab import files
    print('Upload diyarai_lora_dataset.zip from your laptop:')
    uploaded = files.upload()

with zipfile.ZipFile('/content/diyarai_lora_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')
print('✅ Dataset Extracted! Images ready for training.')

In [ ]:
# Step 3: Run 1-Click SDXL LoRA Training (Trigger Word: diyarai)
%cd /content/sd-scripts
!accelerate launch --num_cpu_threads_per_process 2 sdxl_train_network.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --train_data_dir='/content/dataset' \
  --output_dir='/content/output' \
  --output_name='diyarai_sdxl_lora' \
  --save_model_as=safetensors \
  --resolution='1024,1024' \
  --learning_rate=1e-4 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --max_train_epochs=5 \
  --train_batch_size=1 \
  --mixed_precision='fp16' \
  --save_precision='fp16' \
  --gradient_checkpointing \
  --xformers \
  --bucket_reso_steps=64 \
  --min_bucket_reso=512 \
  --max_bucket_reso=1024

print('🎉 TRAINING COMPLETED!')

In [ ]:
# Step 4: Download your trained LoRA Model
from google.colab import files
import os
lora_path = '/content/output/diyarai_sdxl_lora.safetensors'
if os.path.exists(lora_path):
    print(f'Downloading {lora_path}...')
    files.download(lora_path)
else:
    print('Trained model not found in output directory!')